# Text generatation using IMDB

This notebook demonstrates text generation using a Transformer Decoder model applied to the IMDB movie review dataset. The process involves downloading and preparing the data, defining a Transformer-based model, and then training it to generate text based on a given prompt.

In [ ]:
#downloading the data
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

--2026-08-25 09:10:57--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz’

aclImdb_v1.tar.gz   100%[===================>]  80.23M  43.7MB/s    in 1.8s    

2026-08-25 09:10:59 (43.7 MB/s) - ‘aclImdb_v1.tar.gz’ saved [84125825/84125825]



After downloading, we load the dataset into a `tf.data.Dataset` object. We also clean the text by replacing `<br />` tags with spaces.

### Data Preparation

First, we download and extract the IMDB dataset, which contains movie reviews that we'll use to train our text generation model.

In [ ]:
import tensorflow as tf
from tensorflow import keras
dataset = keras.utils.text_dataset_from_directory(
    directory="aclImdb", label_mode=None, batch_size=256
)
dataset = dataset.map(lambda x: tf.strings.regex_replace(x, "<br />", " "))

Found 100006 files.


Next, we use `TextVectorization` to convert our text data into numerical sequences. This layer will build a vocabulary from our dataset and map words to integer indices.

In [ ]:
from tensorflow.keras.layers import TextVectorization

sequence_length = 100
vocab_size = 15000
text_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

text_vectorization.adapt(dataset)

We define a function to prepare the language model dataset. For a language model, we want to predict the next word given the previous words. So, for each sequence, `x` will be the input (all tokens except the last), and `y` will be the target (all tokens except the first, shifted by one position).

In [ ]:
def prepare_lm_dataset(text_batch):
    vectorized_sequences = text_vectorization(text_batch)
    x = vectorized_sequences[:, :-1]
    y = vectorized_sequences[:, 1:]
    return x, y


lm_dataset = dataset.map(prepare_lm_dataset, num_parallel_calls=4)

### Helper Functions

The `reweight_distribution` function is used during text generation to control the randomness of the next token prediction. A lower `temperature` makes the model more deterministic (picks more probable words), while a higher `temperature` makes it more creative (picks less probable words).

In [ ]:
import numpy as np
def reweight_distribution(original_distribution, temperature=0.5):
  distribution = np.log(original_distribution) / temperature
  distribution = np.exp(distribution)
  return distribution / np.sum(distribution)

### Model Definition

We define a `TransformerDecoder` block, which is a core component of our text generation model. It includes self-attention, layer normalization, and a feed-forward network.

In [ ]:
from keras import layers

class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(
            num_heads, key_dim, dropout=0.1
        )
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()
        self.dropout = layers.Dropout(0.1)

    def call(self, inputs):
        residual = x = inputs
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = self.dropout(x)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = self.dropout(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

The `PositionalEmbedding` layer combines token embeddings with positional embeddings. Positional embeddings are crucial in transformers to provide information about the order of words in a sequence, as self-attention layers are otherwise position-agnostic.

In [ ]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs, reverse=False):
        if reverse:
            token_embeddings = self.token_embeddings.embeddings
            return ops.matmul(inputs, ops.transpose(token_embeddings))
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

Now, we build the complete text generation model. It takes an input sequence, applies positional embedding, passes it through the `TransformerDecoder` block, and finally uses a dense layer with a softmax activation to output probabilities for each word in the vocabulary.

In [ ]:
from tensorflow.keras import layers
embed_dim = 256
latent_dim = 2048
num_heads = 2
inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(loss="sparse_categorical_crossentropy", optimizer="rmsprop")

The `sample_next` function helps us to sample the next word from the model's predicted probability distribution, taking into account the `temperature` parameter for controlling randomness.

In [ ]:
tokens_index = dict(enumerate(text_vectorization.get_vocabulary()))
def sample_next(predictions, temperature=1.0):
  predictions = np.asarray(predictions).astype("float64")
  predictions = np.log(predictions) / temperature
  exp_preds = np.exp(predictions)
  predictions = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1, predictions, 1)
  return np.argmax(probas)

### Text Generation Callback

We define a custom Keras callback, `TextGenerator`, to generate text at the end of each training epoch. This allows us to observe the model's progress in generating coherent and contextually relevant text.

In [ ]:
#the text  generator callback
class TextGenerator(keras.callbacks.Callback):
  def __init__(self, prompt, generate_length, model_input_length, temperatures=(1.,), print_freq=1):
    self.prompt = prompt
    self.generate_length = generate_length
    self.model_input_length = model_input_length
    self.temperatures = temperatures
    self.print_freq = print_freq

  def on_epoch_end(self, epoch, logs=None):
    if (epoch + 1) % self.print_freq != 0:
      return
    for temperature in self.temperatures:
      print("== Generating with temperature", temperature)
      sentence = self.prompt
      for i in range(self.generate_length):
        tokenized_sentence = text_vectorization([sentence])
        predictions = self.model(tokenized_sentence)
        next_token = sample_next(predictions[0, i, :])
        sampled_token = tokens_index[next_token]
        sentence += " " + sampled_token
      print(sentence)

Here we initialize the `TextGenerator` callback with a starting prompt, the desired length of the generated text, the model's input sequence length, and a range of temperatures to observe different generation styles.

In [ ]:
prompt = "This movie"
text_gen_callback = TextGenerator(
    prompt,
    generate_length=50,
    model_input_length=sequence_length,
    temperatures=(0.2, 0.5, 0.7, 1., 1.5))

### Model Training

Finally, we train the model using our prepared language modeling dataset. The `TextGenerator` callback will print generated text at the end of each epoch, allowing us to see how the model learns to generate human-like text.

In [ ]:
model.fit(lm_dataset, epochs=200, callbacks=[text_gen_callback])

Epoch 1/200
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - loss: 6.1880== Generating with temperature 0.2
This movie site is is not a one car its [UNK] a environment good however cast it of just boxing see hes its got easy one entertainment that so if i you thought enemy ive but seen i a was lot like of they what were could painfully be [UNK] allowed unwatchable
== Generating with temperature 0.5
This movie movie or was watched all it the for last something message shows bad your as blood it resulted is she a saw error 90 about missing more hype good that comments others it this might has have a engage locals [UNK] and the his capsule don for felt [UNK] so
== Generating with temperature 0.7
This movie movie first not this sure dvd in i my did time not now originally ive thought seen i it got is all scott would teacher mention if a these good [UNK] frankenstein and of you my somehow friends powered from on par so before yes there the was basketball supposed
== Generating with temperature 

### Generated Text Summary (Epoch 105)

Here are examples of text generated by the model with different temperatures, illustrating how temperature affects creativity and coherence:

**Temperature 0.2 (Less creative, more deterministic):**

"This movie is is about no two one people is in killing it women takes are us called to smooth a and nice handed girls plot and but [UNK] everything man is is wrong he with turns her his into opponent this and could boring be for a murder b by they"

**Temperature 1.0 (More balanced creativity):**

"This movie is begins horrendous as this ridiculous is [UNK] there and is [UNK] no best suspense i thriller have and ever enjoyable seen 80s in italian la cinema beyond long the boring [UNK] and time hailed is as disgusting full and of repulsive [UNK] [UNK] from by start the to two"

**Temperature 1.5 (More creative, less coherent):**

"This movie is was stupid very and very weird stupid its for funny what scenes really with some angel really and desperate [UNK] attempts there at were acting just originally as a a stunt bimbo why while do it you keeps get on a watching low identical budget uniform with with a"